In [0]:
-- CineClub ticket breakdown vs Total Business (tickets + ticket revenue)
-- Scope: Cineplex theatres (BR-010, BR-011), ticket lines (BR-016)
-- Date:  Transaction_DateTime (TL-04). Period: Jan–Sep 2025 and Jan–Sep 2026
CREATE OR REPLACE TEMP VIEW v_theatre_ticket_lines AS
SELECT
    CAST(DATE_TRUNC('MONTH', t.Transaction_DateTime) AS DATE) AS report_month,
    t.CineClubFLAG,
    t.CineClub_Discount_Type_ID,
    CASE
        WHEN t.CineClubFLAG = 'Yes' THEN
            CASE t.CineClub_Discount_Type_ID          -- BR-018
                WHEN 1 THEN 'CC Member, no discount'
                WHEN 6 THEN 'CC Redeemed ($9.99)'
                WHEN 8 THEN 'CC Redeemed ($10.99)'
                WHEN 5 THEN 'CC Companion ($9.99)'
                WHEN 7 THEN 'CC Companion ($10.99)'
                ELSE        'CC Other / unmapped'
            END
        ELSE 'Non-CineClub'
    END AS ticket_category,
    t.Quantity,
    t.Net_Revenue
FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
JOIN cpx_dataanalytics_gold.customer360.cpx_d_location loc
  ON t.LocationId = loc.LocationID
WHERE loc.LC_Location_Type_Description = 'Cineplex Theatre'
  AND t.RecRoomFLAG = 'No'
  AND t.STOREFLAG   = 'No'
  AND t.TicketFLAG  = 'Yes'
  AND (   (t.Transaction_DateTime >= TIMESTAMP '2025-01-01' AND t.Transaction_DateTime < TIMESTAMP '2025-10-01')
       OR (t.Transaction_DateTime >= TIMESTAMP '2026-01-01' AND t.Transaction_DateTime < TIMESTAMP '2026-10-01'));

WITH category_totals AS (
    -- Monthly
    SELECT DATE_FORMAT(report_month, 'yyyy-MM') AS period, ticket_category,
           SUM(Quantity)    AS tickets,      -- refunds net out (KPI-09)
           SUM(Net_Revenue) AS revenue
    FROM v_theatre_ticket_lines
    GROUP BY DATE_FORMAT(report_month, 'yyyy-MM'), ticket_category

    UNION ALL
    -- YTD (Jan–Sep of each year)
    SELECT CONCAT('YTD ', YEAR(report_month)) AS period, ticket_category,
           SUM(Quantity), SUM(Net_Revenue)
    FROM v_theatre_ticket_lines
    GROUP BY CONCAT('YTD ', YEAR(report_month)), ticket_category
),
report_rows AS (
    SELECT period, ticket_category, tickets, revenue
    FROM category_totals
    WHERE ticket_category <> 'Non-CineClub'

    UNION ALL
    SELECT period, 'CineClub Total', SUM(tickets), SUM(revenue)
    FROM category_totals
    WHERE ticket_category <> 'Non-CineClub'
    GROUP BY period

    UNION ALL
    -- Surcharge = Net revenue of $9.99/$10.99 tickets − qty × ticket value
    SELECT period, 'CC Surcharge Revenue',
           SUM(tickets),
           SUM(revenue)
             - SUM(CASE WHEN ticket_category IN ('CC Redeemed ($9.99)', 'CC Companion ($9.99)')
                        THEN tickets * 9.99 ELSE 0 END)
             - SUM(CASE WHEN ticket_category IN ('CC Redeemed ($10.99)', 'CC Companion ($10.99)')
                        THEN tickets * 10.99 ELSE 0 END)
    FROM category_totals
    WHERE ticket_category IN ('CC Redeemed ($9.99)',  'CC Companion ($9.99)',
                              'CC Redeemed ($10.99)', 'CC Companion ($10.99)')
    GROUP BY period

    UNION ALL
    SELECT period, 'Total Business', SUM(tickets), SUM(revenue)
    FROM category_totals
    GROUP BY period
)
SELECT
    period,
    ticket_category,
    tickets,
    ROUND(revenue, 2) AS revenue,
    ROUND(100 * tickets / NULLIF(MAX(CASE WHEN ticket_category = 'Total Business' THEN tickets END)
                                 OVER (PARTITION BY period), 0), 2) AS pct_total_biz_tickets,
    ROUND(100 * revenue / NULLIF(MAX(CASE WHEN ticket_category = 'Total Business' THEN revenue END)
                                 OVER (PARTITION BY period), 0), 2) AS pct_total_biz_revenue
FROM report_rows
ORDER BY period,                       -- months first, then YTD 2025, YTD 2026
         CASE ticket_category
             WHEN 'Total Business'         THEN 1
             WHEN 'CineClub Total'         THEN 2
             WHEN 'CC Member, no discount' THEN 3
             WHEN 'CC Redeemed ($9.99)'    THEN 4
             WHEN 'CC Redeemed ($10.99)'   THEN 5
             WHEN 'CC Companion ($9.99)'   THEN 6
             WHEN 'CC Companion ($10.99)'  THEN 7
             WHEN 'CC Surcharge Revenue'   THEN 8
             ELSE 9
         END;